# 가상 참여자 노트북 (정석: 중복 제거 + 랜덤 포레스트 + CV)

In [1]:
import pandas as pd
import numpy as np
import joblib
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)

(1279, 13) (320, 12)


In [2]:
def display_scores(scores):
    print('Scores:', scores)
    print('Mean:', scores.mean())
    print('Standard deviation:', scores.std())

In [3]:
before = len(train)
train = train.drop_duplicates(subset=[c for c in train.columns if c not in ('id', 'quality')])
print('같은 와인 행 제거:', before, '->', len(train))
X, y = train.drop(columns=['id', 'quality']), train['quality']
X_test = test.drop(columns=['id'])

같은 와인 행 제거: 1279 -> 1083


In [4]:
from sklearn.model_selection import KFold, cross_val_score
cv = KFold(n_splits=10, shuffle=True, random_state=0)

In [5]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=300, random_state=0)

In [6]:
scores = np.sqrt(-cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv))
display_scores(scores)
model.fit(X, y)
pred = model.predict(X_test)

Scores: [0.64464808 0.67021761 0.66713966 0.67061715 0.66293927 0.65739784
 0.66814612 0.69637502 0.63757441 0.55811733]
Mean: 0.6533172502604694
Standard deviation: 0.03513557261226841


In [7]:
pd.DataFrame({'id': test['id'], 'quality': pred}).to_csv('submission.csv', index=False)
joblib.dump(model, 'model.pkl')

['model.pkl']